# P0 · Anomaly replication on 3CAD (preregistered)

A dataset neither DINOv2 nor PatchCore was developed on: 3CAD (Yang et al., AAAI 2025), 27,039 images of
3C-product parts from real production lines, 8 categories, 47 defect types. Settings and predictions are
fixed in [`docs/preregistration_3cad.md`](https://github.com/M3R2T2L2/bovw_forensics/blob/main/docs/preregistration_3cad.md),
identical to VisA. **Primary image score: mean of the top-10 patch scores.**

| ID | Prediction |
|---|---|
| P1 | codebook AUPRO > coreset AUPRO at M = 64, 256, 1024 |
| P2 | codebook top-10 image AUROC >= coreset - 0.005 at M = 1024 |
| P3 | codebook pixel AUROC >= coreset at M = 1024 |
| P4 | codebook within 0.01 of the full bank at K = 1024 (image top-10, pixel, AUPRO) |

**Licence:** none stated by the authors; research use, confirm before publishing.
**Runtime (T4):** about 3–4 h. Features are cached on the Colab disk (too large for Drive); results go to Drive.
After a disconnect, run everything again: finished categories are skipped. A High-RAM runtime is safer if you have one.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data
Downloads the official archive from Google Drive with `gdown` (or uses a copy you placed in `MyDrive/bovw-forensics/data/`).

In [ ]:
import os, glob, shutil, subprocess, torch
if not torch.cuda.is_available():   # check before the long download
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
from bovw import ad_data
LOCAL = "/content/3cad"
DRIVE_COPY = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
ARCHIVE = "/content/3cad_archive"

if not os.path.isdir(LOCAL):
    if DRIVE_COPY:
        shutil.copy(DRIVE_COPY[0], ARCHIVE)
    else:
        import gdown
        out = gdown.download(id=ad_data.THREECAD_GDRIVE_ID, output=ARCHIVE, quiet=False)
        if out is None:
            raise SystemExit("gdown could not fetch the archive (Drive quota?). Open "
                             "https://drive.google.com/file/d/" + ad_data.THREECAD_GDRIVE_ID +
                             " , download it, upload to MyDrive/bovw-forensics/data/ as 3CAD.<ext>, and rerun.")
    os.makedirs(LOCAL)
    magic = open(ARCHIVE, "rb").read(4)
    fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
    shutil.unpack_archive(ARCHIVE, LOCAL, fmt)
    print("archive SHA-256 (record in the paper):", ad_data.sha256sum(ARCHIVE))
    os.remove(ARCHIVE)

ROOT = ad_data.find_root(LOCAL, "Copper_Stator")
for c in ad_data.THREECAD_CATEGORIES:
    got, want = ad_data.count_mvtec_style(ROOT, c), ad_data.THREECAD_COUNTS[c]
    print(f"{c:<28} train/test {got}  expected {want}  {'OK' if got == want else 'MISMATCH'}")

## 3 · Run

In [ ]:
from bovw.sweep import load_config
from bovw import anomaly

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all. "
                     "(On CPU, feature extraction takes hours per category.)")
print("GPU:", torch.cuda.get_device_name(0))

cfg = load_config("configs/p0_anomaly_3cad.yaml")
cfg["data"]["root"] = ROOT
df = anomaly.run(cfg)

## 4 · Preregistered checks

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
checks = anomaly.prereg_checks(df)
checks.to_csv(f"{cfg['results_dir']}/{cfg['name']}_prereg.csv", index=False)
checks

## 5 · Full table (secondary metrics are exploratory)

In [ ]:
t = anomaly.followup_table(df)
t.to_csv(f"{cfg['results_dir']}/{cfg['name']}_followup.csv")
t.round(3)

In [ ]:
d = df[df.k.fillna(0).astype(int).isin([0, 1024])]
pd.concat({m: d.pivot_table(index="category", columns="method", values=m)
           for m in ["image_auroc_top10", "pixel_auroc", "aupro"]}, axis=1).round(3)